# DejaVu — Anticipated Scenario Dataset

Carrega o CSV gerado pelo `AntecipatedScenarioDatasetRecorder` e exibe o DataFrame.

---

## Dicionário de colunas (97 colunas)

### 1. Identidade do tick

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `episode` | int | Número do episódio. Cada episódio é uma execução completa da tarefa do zero. |
| `step` | int | Número do tick dentro do episódio. Incrementado a cada percepção recebida do simulador. |

---

### 2. Parâmetros monitorados pelo ASM (State Machine)

Derivados dos sensores brutos pelo `MonitorARM` e usados diretamente como guards/condições na State Machine do DejaVu. Todos são inteiros para compatibilidade com guards booleanos da SM.

| Coluna | Tipo | Como é calculado | Descrição |
|--------|------|-----------------|-----------|
| `task_started` | int (0/1) | Hardcoded = 1 | Tarefa iniciada. Sempre 1 enquanto o episódio corre. |
| `object_available` | int (0/1) | Hardcoded = 1 | Objeto disponível na cena. Sempre 1 neste cenário. |
| `gripper_width_cm` | int | `int(fingers_width * 100)` | Abertura da garra em cm. Aberta ≥ 7.5 cm, fechada ≤ 6 cm (histerese entre 6 e 7.5). |
| `distance_ee_object_cm` | int | `max(0, (dist_ee_to_cube − 0.025) × 100)` | Distância do end-effector ao ponto de contato do cubo em cm. Desconta 2.5 cm do comprimento dos dedos do Franka Panda. |
| `grasp_completed` | int (0/1) | `1 se fingers_width ≤ 0.060 m` | Garra considerada fechada sobre o objeto. Histerese: só reabre se fingers_width > 0.075 m. |
| `finger_contacts` | int (0–2) | Lido direto do simulador | Número de dedos em contato físico com o objeto. |
| `grasp_attempts` | int | Contador acumulativo por episódio | Quantas vezes a garra fechou (transições 0→1 em `grasp_completed`) desde o início do episódio. |
| `object_lift_height_cm` | int | `max(0, (cube_z − cube_z_inicial) × 100)` | Altura do cubo acima da posição inicial em cm. Zero até o cubo ser levantado. |
| `distance_object_goal_cm` | int | `norm(cube_xy − target_xy) × 100` | Distância XY (plano horizontal) entre o cubo e o target em cm. Ignora a componente Z. |
| `task_aborted` | int (0/1) | `1 se current_task == "ABORT" AND is_success` | Tarefa abortada de forma controlada. Normalmente 0. |

---

### 3. Sensores brutos — escalares

Leituras diretas do simulador, antes de qualquer abstração.

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `fingers_width` | float | metros | Abertura real da garra medida entre os dois dedos. |
| `dist_ee_to_cube` | float | metros | Distância 3D do end-effector ao centro geométrico do cubo. |
| `dist_cube_to_target` | float | metros | Distância 3D do cubo à posição do target (inclui Z). |
| `obstacle_in_path` | bool | — | True se algum obstáculo está no caminho do end-effector. |
| `obstacle_count_in_path` | int | — | Número de obstáculos detectados no caminho. |
| `reward` | float | — | Recompensa do agente RL neste tick. Tipicamente negativo e proporcional à distância ao objetivo. |
| `is_success` | bool | — | True se a tarefa foi concluída com sucesso. Sinaliza fim do episódio. |

---

### 4. End-effector — posição e velocidade

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `ee_x`, `ee_y`, `ee_z` | float | metros | Posição 3D do end-effector (ponta da garra) no espaço cartesiano. |
| `ee_vx`, `ee_vy`, `ee_vz` | float | m/s | Velocidade linear 3D do end-effector. |

---

### 5. Cubo — posição, orientação e velocidade

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `cube_x`, `cube_y`, `cube_z` | float | metros | Posição 3D do centro do cubo. |
| `cube_roll`, `cube_pitch`, `cube_yaw` | float | radianos | Orientação do cubo (ângulos de Euler). |
| `cube_vx`, `cube_vy`, `cube_vz` | float | m/s | Velocidade linear do cubo. Não nula quando o cubo está sendo movido. |

---

### 6. Target (objetivo)

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `target_x`, `target_y`, `target_z` | float | metros | Posição 3D da posição-alvo onde o cubo deve ser colocado. Constante durante o episódio. |

---

### 7. Ação do agente RL

Saída da política do agente de Reinforcement Learning neste tick, antes de ser executada.

| Coluna | Tipo | Range | Descrição |
|--------|------|-------|-----------|
| `action_x`, `action_y`, `action_z` | float | [−1, 1] | Deslocamento comandado ao end-effector nos eixos X, Y, Z. |
| `action_gripper` | float | [−1, 1] | Comando da garra: +1 = abrir, −1 = fechar. |

---

### 8. Ângulos das juntas do robô

7 juntas do Franka Panda, da base até o pulso.

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `j0`–`j6` | float | radianos | Ângulo atual de cada junta. |
| `jv0`–`jv6` | float | rad/s | Velocidade angular atual de cada junta. |

---

### 9. Contexto da tarefa

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `current_task` | str | Nome da tarefa de alto nível. Ex: `OBJECT_DELIVERY_SEQUENCE`. |
| `current_subtask` | str | Subtarefa ativa no Managing neste tick. Valores: `APPROACH_OBJECT`, `GRASP_OBJECT`, `RETRY_GRASP`, `SAFE_ABORT`, `LIFT_OBJECT`, `TRANSPORT_OBJECT`, `PLACE_OBJECT`. |
| `active_target_name` | str | Nome do target ativo. Ex: `"target"`. |

---

### 10. Objetos da cena (flattenado)

Propriedades do cubo conforme definido na configuração do episódio. Constantes durante o episódio, exceto `current_position`.

| Coluna | Descrição |
|--------|-----------|
| `objects.object_1.type` | Tipo do objeto. Ex: `"box"`. |
| `objects.object_1.size.0/1/2` | Dimensões XYZ em metros. |
| `objects.object_1.mass` | Massa em kg. |
| `objects.object_1.color.0/1/2/3` | Cor RGBA, cada componente em [0, 1]. |
| `objects.object_1.initial_position.0/1/2` | Posição XYZ inicial do cubo em metros. |
| `objects.object_1.lateral_friction` | Coeficiente de atrito lateral. |
| `objects.object_1.spinning_friction` | Coeficiente de atrito rotacional. |
| `objects.object_1.current_position.0/1/2` | Posição XYZ atual do cubo (maior precisão que `cube_x/y/z`). |

---

### 11. Cena (flattenado)

| Coluna | Descrição |
|--------|-----------|
| `scene.table.length`, `scene.table.width`, `scene.table.height` | Dimensões da mesa em metros. |
| `scene.table.x_offset` | Offset X da mesa em relação à origem. |
| `scene.table.lateral_friction`, `scene.table.spinning_friction` | Atrito da superfície da mesa. |

---

### 12. Configuração do robô (flattenado)

| Coluna | Descrição |
|--------|-----------|
| `robot_config.control_type` | Tipo de controle: `"ee"` (end-effector space) ou `"joint"`. |
| `robot_config.block_gripper` | True se a garra está travada e não responde a comandos. |
| `robot_config.base_position.0/1/2` | Posição XYZ da base do robô em metros. |

---

### 13. Objetivo da tarefa (flattenado)

| Coluna | Descrição |
|--------|-----------|
| `target_goal.type` | Tipo de sequência. Ex: `"goal_sequence"`. |
| `target_goal.targets.0.name` | Nome do target. Ex: `"target"`. |
| `target_goal.targets.0.position.0/1/2` | Posição XYZ do target em metros. |
| `target_goal.mode` | Modo de execução. Ex: `"goal_sequence"`. |

---

### 14. Scripts de comportamento (flattenado)

| Coluna | Descrição |
|--------|-----------|
| `scripts.script_1` | Script de comportamento 1 ativo (bool). |
| `scripts.reach_only` | True se o episódio é de apenas aproximação (sem pegar). |
| `scripts.left_right` | True se o modo de movimento lateral está ativado. |

---

### 15. Monitoramento DejaVu ⭐

Colunas produzidas exclusivamente pelo DejaVu. São o **output do monitoramento** — não existem no Managing nem no simulador.

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `active_state` | str | Estado ativo na State Machine do DejaVu após processar o tick. Segue o fluxo: `INIT → PHI_1 → S2 → PHI_3 → S4 → PHI_5 → S6 → PHI_7 → S8 → ... → FINAL`. Estados `PHI_N` são de verificação (transientes), estados `SN` são estáveis (aguardando próxima ação), `ERR_N` indicam violação, `FINAL` = tarefa concluída. |
| `sat` | bool / None | **Label do dataset.** `True` = comportamento confirmado como antecipado (SM avançou sem erro). `False` = cenário não antecipado detectado (SM travou ou chegou em estado ERR). `None` = tick de monitoramento passivo (nenhuma transição de subtarefa ocorreu neste tick). |

In [7]:
import pandas as pd

CSV_PATH = r"C:\Users\lucas_alves\Workspace\self-adaptive-arm-simulator\3-dejavu\output\arm\antecipated_scenario_dataset\antecipated_scenario_dataset_20260812_234121.csv"

df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape[0]} linhas × {df.shape[1]} colunas")
df[["step", "task_started", "gripper_width_cm","distance_ee_object_cm", "grasp_completed", "finger_contacts", "grasp_attempts", "object_lift_height_cm", "distance_object_goal_cm", "reward", "is_success", "current_subtask","active_state","sat"]]

Shape: 69 linhas × 97 colunas


,step,task_started,gripper_width_cm,distance_ee_object_cm,grasp_completed,finger_contacts,grasp_attempts,object_lift_height_cm,distance_object_goal_cm,reward,is_success,current_subtask,active_state,sat
0,1,1,7,28,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,NaN
1,2,1,7,25,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,NaN
2,3,1,7,22,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,NaN
3,4,1,7,19,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,NaN
4,5,1,7,16,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,65,1,7,0,0,0,1,0,1,-0.013007,False,PLACE_OBJECT,S26,True
65,66,1,7,0,0,0,1,0,1,-0.013005,False,PLACE_OBJECT,S26,True
66,67,1,7,0,0,0,1,0,1,-0.013005,False,PLACE_OBJECT,S26,True
67,68,1,7,0,0,0,1,0,1,-0.013005,False,PLACE_OBJECT,S26,True
